# 01 – Dataset Exploration

This notebook explores the raw brain MRI dataset:
- Dataset statistics (size, tumour prevalence)
- Sample image/mask visualisation
- Intensity distribution analysis
- Train/val/test split preview

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import yaml
import cv2

from data.preprocessing import Preprocessor
from data.splitter import discover_pairs, split_dataset

%matplotlib inline
plt.rcParams['figure.facecolor'] = '#0c1a3a'
plt.rcParams['axes.facecolor']   = '#0c1a3a'
plt.rcParams['text.color']       = 'white'
plt.rcParams['axes.labelcolor']  = 'white'
plt.rcParams['xtick.color']      = 'white'
plt.rcParams['ytick.color']      = 'white'

In [ ]:
with open('../configs/config.yaml') as f:
    cfg = yaml.safe_load(f)

data_cfg = cfg['data']
prep_cfg = {**data_cfg, **cfg.get('preprocessing', {})}

raw_dir   = Path('../') / data_cfg['raw_dir']
img_dir   = raw_dir / data_cfg['images_subdir']
msk_dir   = raw_dir / data_cfg['masks_subdir']

print(f'Images dir: {img_dir}')
print(f'Masks  dir: {msk_dir}')

In [ ]:
img_paths, msk_paths = discover_pairs(img_dir, msk_dir)
print(f'Discovered {len(img_paths)} image–mask pairs')

preprocessor = Preprocessor(prep_cfg)

# Tumour prevalence
tumor_count = sum(
    1 for m in msk_paths
    if preprocessor.load_mask(m).max() > 0
)
print(f'Tumour present: {tumor_count} / {len(msk_paths)} ({100*tumor_count/len(msk_paths):.1f}%)')

In [ ]:
# Plot first 8 samples
n = min(8, len(img_paths))
fig, axes = plt.subplots(2, n, figsize=(3*n, 6))
fig.suptitle('Sample MRI Scans + Masks', color='white', fontsize=14)

for i in range(n):
    img  = preprocessor.load_image(img_paths[i])
    mask = preprocessor.load_mask(msk_paths[i])

    axes[0, i].imshow(img.squeeze(), cmap='gray')
    axes[0, i].set_title(img_paths[i].stem[:12], fontsize=8)
    axes[0, i].axis('off')

    axes[1, i].imshow(mask.squeeze(), cmap='hot')
    axes[1, i].set_title('mask', fontsize=8)
    axes[1, i].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# Intensity distribution
sample_intensities = []
for p in img_paths[:50]:
    img = preprocessor.load_image(p).astype(np.float32)
    sample_intensities.extend(img.flatten().tolist())

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(sample_intensities, bins=100, color='#2d8eff', alpha=0.7, edgecolor='none')
ax.set_title('Pixel Intensity Distribution (first 50 images)', color='white')
ax.set_xlabel('Pixel value')
ax.set_ylabel('Count')
plt.tight_layout()
plt.show()

In [ ]:
# Split preview
(tr, _), (va, _), (te, _) = split_dataset(
    img_paths, msk_paths,
    train_ratio   = data_cfg['train_ratio'],
    val_ratio     = data_cfg['val_ratio'],
    test_ratio    = data_cfg['test_ratio'],
    patient_level = data_cfg['patient_level_split'],
    separator     = data_cfg['patient_id_separator'],
    id_index      = data_cfg['patient_id_index'],
)
print(f'Train: {len(tr)} | Val: {len(va)} | Test: {len(te)}')

labels = ['Train', 'Val', 'Test']
sizes  = [len(tr), len(va), len(te)]
colors = ['#2d8eff', '#7c3aed', '#22c55e']

fig, ax = plt.subplots(figsize=(5, 5))
ax.pie(sizes, labels=labels, colors=colors, autopct='%1.1f%%',
       textprops={'color': 'white'}, startangle=140)
ax.set_title('Dataset Split', color='white')
plt.tight_layout()
plt.show()